## Data

### 1. Load the pinned snapshot

In [11]:
import json
from pathlib import Path

ROOT = Path.cwd()
ROOT = ROOT.parent if ROOT.name == "research" else ROOT
config = json.loads((ROOT / "research" / "source_config.json").read_text())
snapshot = ROOT / "data" / "raw" / "statsbomb" / config["source_ref"]
catalog_path = (
    snapshot
    / "matches"
    / str(config["competition_id"])
    / f"{config['season_id']}.json"
)
matches = json.loads(catalog_path.read_text())
SHOOTOUT_PERIOD = 5

print("Source scope")
print(json.dumps({
    "source_ref": config["source_ref"],
    "competition_id": config["competition_id"],
    "season_id": config["season_id"],
    "catalog_path": str(catalog_path.relative_to(ROOT)),
    "matches_observed": len(matches),
}, indent=2))

Source scope
{
  "source_ref": "b0bc9f22dd77c206ddedc1d742893b3bbe64baec",
  "competition_id": 55,
  "season_id": 282,
  "catalog_path": "data/raw/statsbomb/b0bc9f22dd77c206ddedc1d742893b3bbe64baec/matches/55/282.json",
  "matches_observed": 51
}


### 2. Inspect match fields and participant keys

In [12]:
def field_profile(field_counts, record_count):
    return {
        "records": record_count,
        "observed_on_every_record": [
            field
            for field, count in sorted(field_counts.items())
            if count == record_count
        ],
        "optional_or_type_specific": {
            field: count
            for field, count in sorted(field_counts.items())
            if count < record_count
        },
    }


match_field_counts = {}
for match in matches:
    for field in match:
        match_field_counts[field] = match_field_counts.get(field, 0) +1

match_ids = [match.get("match_id") for match in matches]
match_ids_present = [
    match_id
    for match_id in match_ids
    if match_id is not None
]
missing_match_ids = len(match_ids) - len(match_ids_present)
duplicate_match_ids = (
    len(match_ids_present) - len(set(match_ids_present))
)

participants = {}
invalid_match_participants = 0
for match in matches:
    match_id = match.get("match_id")
    team_ids = {
        (match.get("home_team") or {}).get("home_team_id"),
        (match.get("away_team") or {}).get("away_team_id"),
    }
    if match_id is not None:
        participants[match_id] = team_ids
    if len(team_ids) != 2 or None in team_ids:
        invalid_match_participants += 1

match_fields = field_profile(match_field_counts, len(matches))
match_checks = {
    "missing_match_ids": missing_match_ids,
    "duplicate_match_ids": duplicate_match_ids,
    "invalid_match_participants": invalid_match_participants,
}

print("Match catalog scan")
print(json.dumps({
    "fields": match_fields,
    "checks": match_checks,
}, indent=2))

Match catalog scan
{
  "fields": {
    "records": 51,
    "observed_on_every_record": [
      "away_score",
      "away_team",
      "competition",
      "competition_stage",
      "home_score",
      "home_team",
      "kick_off",
      "last_updated",
      "last_updated_360",
      "match_date",
      "match_id",
      "match_status",
      "match_status_360",
      "match_week",
      "metadata",
      "referee",
      "season",
      "stadium"
    ],
    "optional_or_type_specific": {}
  },
  "checks": {
    "missing_match_ids": 0,
    "duplicate_match_ids": 0,
    "invalid_match_participants": 0
  }
}


## Results

### 3. Prove event-file and MatchID linkage

## edit

### 3. Check event keys, teams, event types, and shots

In [13]:
event_directory = snapshot / "events"
event_ids = []
event_keys = []
event_field_counts = {}
event_type_counts = {}
shot_field_counts = {}

event_files_parsed = 0
event_list_roots = 0
events_with_embedded_match_id = 0
shot_count = 0
non_shootout_shots = 0
non_shootout_expected_goals = 0.0

missing_event_files = 0
event_roots_not_lists = 0
empty_event_files = 0
non_object_events = 0
bad_team_references = 0
shots_missing_detail = 0
shots_missing_expected_goals = 0
invalid_shot_expected_goals = 0

for match_id, valid_teams in participants.items():
    event_path = event_directory / f"{match_id}.json"
    if not event_path.exists():
        missing_event_files += 1
        continue

    events = json.loads(event_path.read_text())
    event_files_parsed += 1
    if not isinstance(events, list):
        event_roots_not_lists += 1
        continue

    event_list_roots += 1
    if not events:
        empty_event_files += 1

    for event in events:
        if not isinstance(event, dict):
            non_object_events += 1
            continue

        if "match_id" in event:
            events_with_embedded_match_id += 1

        for field in event:
            event_field_counts[field] = (
                event_field_counts.get(field, 0) + 1
            )

        event_id = event.get("id")
        event_index = event.get("index")
        event_ids.append(event_id)
        event_keys.append((match_id, event_index))

        event_type = (event.get("type") or {}).get("name")
        if event_type:
            event_type_counts[event_type] = (
                event_type_counts.get(event_type, 0) + 1
            )

        period = event.get("period")
        team_id = (event.get("team") or {}).get("id")
        if team_id not in valid_teams:
            bad_team_references += 1

        if event_type == "Shot":
            shot_count += 1
            shot = event.get("shot")
            if not isinstance(shot, dict):
                shots_missing_detail += 1
                continue

            for field in shot:
                display_field = (
                    "expected_goals"
                    if field == "statsbomb_xg"
                    else field
                )
                shot_field_counts[display_field] = (
                    shot_field_counts.get(display_field, 0) + 1
                )

            expected_goals = shot.get("statsbomb_xg")
            if expected_goals is None:
                shots_missing_expected_goals += 1

            if period != SHOOTOUT_PERIOD:
                non_shootout_shots += 1
                valid_expected_goals = (
                    isinstance(expected_goals, (int, float))
                    and 0 <= expected_goals <= 1
                )
                if not valid_expected_goals:
                    invalid_shot_expected_goals += 1
                else:
                    non_shootout_expected_goals += expected_goals

event_ids_present = [
    event_id
    for event_id in event_ids
    if event_id is not None
]
event_keys_present = [
    event_key
    for event_key in event_keys
    if event_key[1] is not None
]

print("Event scan complete")
print(json.dumps({
    "event_files_expected": len(participants),
    "event_files_parsed": event_files_parsed,
    "event_roots_that_are_lists": event_list_roots,
    "events_observed": len(event_ids),
    "event_types_observed": len(event_type_counts),
    "shots_observed": shot_count,
    "shootout_shots_observed": (
        shot_count - non_shootout_shots
    ),
}, indent=2))

Event scan complete
{
  "event_files_expected": 51,
  "event_files_parsed": 51,
  "event_roots_that_are_lists": 51,
  "events_observed": 187924,
  "event_types_observed": 33,
  "shots_observed": 1340,
  "shootout_shots_observed": 24
}


### 4 answer profiling questions

In [14]:
checks = {
    **match_checks,
    "missing_event_files": missing_event_files,
    "event_roots_not_lists": event_roots_not_lists,
    "empty_event_files": empty_event_files,
    "non_object_events": non_object_events,
    "missing_event_ids": (
        len(event_ids) - len(event_ids_present)
    ),
    "duplicate_event_ids": (
        len(event_ids_present) - len(set(event_ids_present))
    ),
    "missing_event_indexes": (
        len(event_keys) - len(event_keys_present)
    ),
    "duplicate_match_indexes": (
        len(event_keys_present) - len(set(event_keys_present))
    ),
    "bad_team_references": bad_team_references,
    "shots_missing_detail": shots_missing_detail,
    "shots_missing_expected_goals": shots_missing_expected_goals,
    "invalid_shot_expected_goals": invalid_shot_expected_goals,
}

event_fields = field_profile(
    event_field_counts,
    len(event_ids),
)
shot_fields = field_profile(
    shot_field_counts,
    shot_count,
)

relationship_checks = [
    "missing_match_ids",
    "duplicate_match_ids",
    "invalid_match_participants",
    "missing_event_files",
    "event_roots_not_lists",
    "empty_event_files",
    "non_object_events",
    "missing_event_ids",
    "duplicate_event_ids",
    "missing_event_indexes",
    "duplicate_match_indexes",
    "bad_team_references",
]
relationships_valid = not any(
    checks[name]
    for name in relationship_checks
)
dq_failures = {
    name: count
    for name, count in sorted(checks.items())
    if count
}

print("1. Are all fields required, or are some optional?")
print(json.dumps({
    "interpretation": (
        "This snapshot shows fields as opposed to schema, but it will remain unchanged."
    ),
    "match_catalog": match_fields,
    "event_envelope": event_fields,
    "shot_payload": shot_fields,
}, indent = 2))

print(
    "\n\n\n2 can I parse and make relationships between JSON types as expected?"
)
print(json.dumps({
    "relationships_valid": relationships_valid,
    "event_files_expected": len(participants),
    "event_files_parsed": event_files_parsed,
    "event_roots_that_are_lists": event_list_roots,
    "event_records_that_are_objects": len(event_ids),
    "events_with_embedded_match_id": (
        events_with_embedded_match_id
    ),
    "event_team_orphans": checks["bad_team_references"],
    "relationship_method": (
        "The catalog MatchID selects an event filename and must be "
        "attached to those event rows during ingestion."
    ),
}, indent=2))

print("\n\n3. Are there any DQ issues???")
print(json.dumps({
    "issues_found": bool(dq_failures),
    "failures": dq_failures,
    "checks": dict(sorted(checks.items())),
}, indent =2))

1. Are all fields required, or are some optional?
{
  "interpretation": "This snapshot shows field prevalence, not a formal schema.",
  "match_catalog": {
    "records": 51,
    "observed_on_every_record": [
      "away_score",
      "away_team",
      "competition",
      "competition_stage",
      "home_score",
      "home_team",
      "kick_off",
      "last_updated",
      "last_updated_360",
      "match_date",
      "match_id",
      "match_status",
      "match_status_360",
      "match_week",
      "metadata",
      "referee",
      "season",
      "stadium"
    ],
    "optional_or_type_specific": {}
  },
  "event_envelope": {
    "records": 187924,
    "observed_on_every_record": [
      "id",
      "index",
      "minute",
      "period",
      "play_pattern",
      "possession",
      "possession_team",
      "second",
      "team",
      "timestamp",
      "type"
    ],
    "optional_or_type_specific": {
      "50_50": 309,
      "bad_behaviour": 58,
      "ball_receipt": 6